# ClaimsIQ 01 — The MCP Server, Backed by Real Snowflake Queries

**This notebook:** Builds `mcp_snowflake_server.py` — a reusable module
containing the MCP server (six tools, each a real Snowflake query) and
the MCP client class. Every later notebook (LangGraph, CrewAI, SWARM)
imports FROM this file, so there is exactly one implementation of each
tool — the M+N problem from Day 12, solved for real this time.

### Why this is a saved `.py` file, not just notebook cells
In Lab 21, the simplified MCP server lived inside one notebook because
only one agent used it. Here, three separate notebooks need to connect
to the SAME server — so the server needs to exist as an importable
module, not be redefined three times.

## Step 1 — Install & connect

In [1]:
%pip install -q snowflake-connector-python

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("SNOWFLAKE_ACCOUNT"), "Set Snowflake env vars first (see Notebook 00)"
print("Environment ready. Writing mcp_snowflake_server.py next.")

Environment ready. Writing mcp_snowflake_server.py next.


## Step 2 — Write the server module

This cell writes the actual file to disk. Read through it carefully —
this IS the MCP server every agent framework connects to for the rest
of the project.

In [3]:
%%writefile mcp_snowflake_server.py
"""
ClaimsIQ MCP Server — wraps a real Snowflake warehouse as six MCP tools.

Same list_tools() / call_tool() discovery pattern as Lab 21's
SimpleMCPServer, now backed by live SQL instead of a Python dict.
"""
import os
import snowflake.connector


def get_connection():
    return snowflake.connector.connect(
        account=os.environ["SNOWFLAKE_ACCOUNT"],
        user=os.environ["SNOWFLAKE_USER"],
        password=os.environ["SNOWFLAKE_PASSWORD"],
        warehouse=os.environ["SNOWFLAKE_WAREHOUSE"],
        database=os.environ["SNOWFLAKE_DATABASE"],
        schema=os.environ["SNOWFLAKE_SCHEMA"],
        role=os.environ.get("SNOWFLAKE_ROLE"),
    )


class SimpleMCPServer:
    """Same shape as Lab 21 — a tool registry with list_tools()/call_tool()."""

    def __init__(self, name):
        self.name = name
        self._tools = {}

    def tool(self, description):
        def decorator(fn):
            self._tools[fn.__name__] = {"fn": fn, "description": description}
            return fn
        return decorator

    def list_tools(self):
        return [
            {"name": name, "description": meta["description"]}
            for name, meta in self._tools.items()
        ]

    def call_tool(self, name, **kwargs):
        if name not in self._tools:
            return {"error": f"Unknown tool: {name}"}
        return self._tools[name]["fn"](**kwargs)


class SimpleMCPClient:
    """Same shape as Lab 21 — discovers tools at connect(), never hardcodes them."""

    def __init__(self, server):
        self.server = server
        self.available_tools = None

    def connect(self):
        self.available_tools = self.server.list_tools()
        print(f"Connected to '{self.server.name}'. Discovered {len(self.available_tools)} tools.")
        return self.available_tools

    def call_tool(self, name, **kwargs):
        return self.server.call_tool(name, **kwargs)


# ============================================================
# The server instance and its six tools
# ============================================================
claims_server = SimpleMCPServer("claimsiq-snowflake")


def _rows_as_dicts(cursor):
    cols = [c[0].lower() for c in cursor.description]
    return [dict(zip(cols, row)) for row in cursor.fetchall()]


@claims_server.tool(
    "Looks up a customer's profile (name, signup date, risk tier) and their "
    "most recent orders, given a customer ID."
)
def get_customer_profile(customer_id: str) -> dict:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT customer_id, name, signup_date, account_risk_tier "
            "FROM CUSTOMERS WHERE customer_id = %s", (customer_id,)
        )
        profile = _rows_as_dicts(cs)
        if not profile:
            return {"error": f"No customer found with ID {customer_id}"}
        cs.execute(
            "SELECT order_id, product_name, order_value, TO_VARCHAR(order_ts) AS order_ts, is_festive_sale "
            "FROM ORDERS WHERE customer_id = %s ORDER BY order_ts DESC LIMIT 5",
            (customer_id,),
        )
        recent_orders = _rows_as_dicts(cs)
        return {"profile": profile[0], "recent_orders": recent_orders}
    finally:
        conn.close()


@claims_server.tool(
    "Looks up the full details of a claim, joined with its order, given a claim ID."
)
def get_claim_details(claim_id: str) -> dict:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT c.claim_id, c.claim_reason, c.claim_amount, TO_VARCHAR(c.filed_ts) AS filed_ts, c.status, "
            "o.order_id, o.product_name, o.order_value, TO_VARCHAR(o.order_ts) AS order_ts, o.is_festive_sale, o.customer_id "
            "FROM CLAIMS c JOIN ORDERS o ON c.order_id = o.order_id "
            "WHERE c.claim_id = %s", (claim_id,)
        )
        result = _rows_as_dicts(cs)
        return result[0] if result else {"error": f"No claim found with ID {claim_id}"}
    finally:
        conn.close()


@claims_server.tool(
    "Checks fraud signals detected for a customer in the last 30 days, given a customer ID."
)
def check_fraud_signals(customer_id: str) -> list:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT signal_type, severity, TO_VARCHAR(detected_ts) AS detected_ts FROM FRAUD_SIGNALS "
            "WHERE customer_id = %s AND detected_ts >= DATEADD(day, -30, CURRENT_TIMESTAMP()) "
            "ORDER BY detected_ts DESC", (customer_id,)
        )
        return _rows_as_dicts(cs)
    finally:
        conn.close()


@claims_server.tool(
    "Returns a customer's transaction count and total amount in the last 48 hours, "
    "including how many were on unrecognized (not verified) devices."
)
def get_transaction_velocity(customer_id: str) -> dict:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT COUNT(*), SUM(amount) FROM TRANSACTIONS "
            "WHERE customer_id = %s AND txn_ts >= DATEADD(hour, -48, CURRENT_TIMESTAMP())",
            (customer_id,),
        )
        count, total = cs.fetchone()
        cs.execute("SELECT verified_devices FROM CUSTOMERS WHERE customer_id = %s", (customer_id,))
        verified = (cs.fetchone() or [""])[0]
        verified_set = set((verified or "").split(","))
        cs.execute(
            "SELECT device_id FROM TRANSACTIONS WHERE customer_id = %s "
            "AND txn_ts >= DATEADD(hour, -48, CURRENT_TIMESTAMP())", (customer_id,)
        )
        devices = [r[0] for r in cs.fetchall()]
        unrecognized = [d for d in devices if d not in verified_set]
        return {
            "txn_count_48h": count,
            "txn_total_48h": float(total) if total else 0.0,
            "unrecognized_device_txns": len(unrecognized),
        }
    finally:
        conn.close()


@claims_server.tool(
    "Compares a customer's average transaction amount to the overall customer "
    "population's average, over the last 7 days — useful for telling a personally "
    "anomalous spend spike apart from a general trend (e.g. a sale event)."
)
def compare_to_peer_spend(customer_id: str) -> dict:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT AVG(amount) FROM TRANSACTIONS WHERE customer_id = %s "
            "AND txn_ts >= DATEADD(day, -7, CURRENT_TIMESTAMP())", (customer_id,)
        )
        customer_avg = cs.fetchone()[0]
        cs.execute(
            "SELECT AVG(amount) FROM TRANSACTIONS "
            "WHERE txn_ts >= DATEADD(day, -7, CURRENT_TIMESTAMP())"
        )
        peer_avg = cs.fetchone()[0]
        ratio = round(float(customer_avg) / float(peer_avg), 2) if customer_avg and peer_avg else None
        return {
            "customer_avg_7d": float(customer_avg) if customer_avg else 0.0,
            "peer_avg_7d": float(peer_avg) if peer_avg else 0.0,
            "customer_to_peer_ratio": ratio,
        }
    finally:
        conn.close()


ALLOWED_ANALYTICS_TABLES = {"CUSTOMERS", "ORDERS", "CLAIMS", "TRANSACTIONS", "FRAUD_SIGNALS"}


@claims_server.tool(
    "Runs a read-only, row-limited SELECT query against the claims warehouse for "
    "ad-hoc analytics. Only SELECT statements are allowed; results are capped at 50 rows."
)
def run_analytics_query(sql: str) -> dict:
    normalized = sql.strip().rstrip(";")
    if not normalized.upper().startswith("SELECT"):
        return {"error": "Only SELECT statements are allowed."}
    if ";" in normalized:
        return {"error": "Multiple statements are not allowed."}
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(f"{normalized} LIMIT 50")
        return {"rows": _rows_as_dicts(cs)}
    except Exception as e:
        return {"error": f"Query failed: {e}"}
    finally:
        conn.close()


Writing mcp_snowflake_server.py


## Step 3 — Import the module and connect a client

From this point on, every notebook that needs these tools just does
`from mcp_snowflake_server import claims_server, SimpleMCPClient`.

In [4]:
from mcp_snowflake_server import claims_server, SimpleMCPClient

mcp_client = SimpleMCPClient(claims_server)
discovered = mcp_client.connect()
for t in discovered:
    print(f"  - {t['name']}: {t['description'][:70]}...")

Connected to 'claimsiq-snowflake'. Discovered 6 tools.
  - get_customer_profile: Looks up a customer's profile (name, signup date, risk tier) and their...
  - get_claim_details: Looks up the full details of a claim, joined with its order, given a c...
  - check_fraud_signals: Checks fraud signals detected for a customer in the last 30 days, give...
  - get_transaction_velocity: Returns a customer's transaction count and total amount in the last 48...
  - compare_to_peer_spend: Compares a customer's average transaction amount to the overall custom...
  - run_analytics_query: Runs a read-only, row-limited SELECT query against the claims warehous...


## Step 4 — Test every tool against real data

In [5]:
print("get_customer_profile:")
print(mcp_client.call_tool("get_customer_profile", customer_id="CUST99001"))
print()
print("check_fraud_signals:")
print(mcp_client.call_tool("check_fraud_signals", customer_id="CUST99001"))
print()
print("get_transaction_velocity:")
print(mcp_client.call_tool("get_transaction_velocity", customer_id="CUST99001"))
print()
print("compare_to_peer_spend:")
print(mcp_client.call_tool("compare_to_peer_spend", customer_id="CUST99001"))

get_customer_profile:
{'profile': {'customer_id': 'CUST99001', 'name': 'Ananya Rao', 'signup_date': datetime.date(2023, 8, 22), 'account_risk_tier': 'low'}, 'recent_orders': [{'order_id': 'ORD99001', 'product_name': 'BassMax Home Theater System', 'order_value': Decimal('45000.00'), 'order_ts': '56639149-09-16 23:09:28.000', 'is_festive_sale': True}]}

check_fraud_signals:
[{'signal_type': 'new_device', 'severity': 'high', 'detected_ts': '56637780-10-03 01:41:19.000'}, {'signal_type': 'velocity_spike', 'severity': 'high', 'detected_ts': '56637552-08-06 17:27:39.000'}]

get_transaction_velocity:
{'txn_count_48h': 9, 'txn_total_48h': 62566.75, 'unrecognized_device_txns': 3}

compare_to_peer_spend:
{'customer_avg_7d': 6951.86111111, 'peer_avg_7d': 2771.21779329, 'customer_to_peer_ratio': 2.51}


In [6]:
print("get_claim_details:")
print(mcp_client.call_tool("get_claim_details", claim_id="CLM99001"))
print()
print("run_analytics_query (ad hoc):")
print(mcp_client.call_tool("run_analytics_query", sql="SELECT account_risk_tier, COUNT(*) as n FROM CUSTOMERS GROUP BY account_risk_tier"))

get_claim_details:
{'claim_id': 'CLM99001', 'claim_reason': 'Speaker not producing sound', 'claim_amount': Decimal('45000.00'), 'filed_ts': '56639377-11-13 06:09:28.000', 'status': 'pending', 'order_id': 'ORD99001', 'product_name': 'BassMax Home Theater System', 'order_value': Decimal('45000.00'), 'order_ts': '56639149-09-16 23:09:28.000', 'is_festive_sale': True, 'customer_id': 'CUST99001'}

run_analytics_query (ad hoc):
{'rows': [{'account_risk_tier': 'low', 'n': 232}, {'account_risk_tier': 'medium', 'n': 47}, {'account_risk_tier': 'high', 'n': 22}]}


In [7]:
import inspect
print(inspect.getsource(claims_server._tools["get_claim_details"]["fn"]))

@claims_server.tool(
    "Looks up the full details of a claim, joined with its order, given a claim ID."
)
def get_claim_details(claim_id: str) -> dict:
    conn = get_connection()
    try:
        cs = conn.cursor()
        cs.execute(
            "SELECT c.claim_id, c.claim_reason, c.claim_amount, TO_VARCHAR(c.filed_ts) AS filed_ts, c.status, "
            "o.order_id, o.product_name, o.order_value, TO_VARCHAR(o.order_ts) AS order_ts, o.is_festive_sale, o.customer_id "
            "FROM CLAIMS c JOIN ORDERS o ON c.order_id = o.order_id "
            "WHERE c.claim_id = %s", (claim_id,)
        )
        result = _rows_as_dicts(cs)
        return result[0] if result else {"error": f"No claim found with ID {claim_id}"}
    finally:
        conn.close()



## Step 5 — Confirm the guardrail on `run_analytics_query`

This should be REJECTED — a preview of Day 14's security discipline,
applied here.

In [8]:
print(mcp_client.call_tool("run_analytics_query", sql="DELETE FROM CUSTOMERS WHERE 1=1"))

{'error': 'Only SELECT statements are allowed.'}


## Deliverable for this notebook

1. Confirm all six tools returned real data in Step 4, not errors.
2. Confirm Step 5's destructive query was rejected.
3. One paragraph: `mcp_snowflake_server.py` now exists as a real file on
   disk. What would need to change in this file for a SECOND team
   (say, a marketing team wanting customer analytics) to reuse these
   same tools for a completely different agent? What would NOT need to
   change?

## What's next

`02_langgraph_agent.ipynb`, `03_crewai_agent.ipynb`, and
`04_swarm_agent.ipynb` all import this same module and connect as MCP
clients — run this notebook first, every time, since it's what
creates `mcp_snowflake_server.py` on disk.